# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you in loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

> _"This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples."_

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, their field names, and their `@id`s. This will help in selecting data for extraction and processing.


In [ ]:
# List all record sets and their details
print('Available Record Sets:')
for rs in metadata.record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  Description: {rs.description}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    @id: {field.id}\n      name: {field.name}\n      Data type: {field.data_type}\n      Description: {field.description}")
    print('-'*40)
# For quick reference, collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames using their `@id` for further analysis. 

In [ ]:
# Select main record set(s) for extraction
print('Record set IDs:', record_set_ids)
# For demonstration, we pick the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Extract records for all record sets into DataFrames
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"@id: {rs_id} -> shape: {df.shape}, columns: {df.columns.tolist()}")

# Display first few rows of the main DataFrame
if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filtering records, normalizing numeric fields, and grouping data. Reference all columns/fields by `@id`.

In [ ]:
# Example EDA on the main record set
from pandas.api.types import is_numeric_dtype

# Find a numeric field (by @id)
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
print('Numeric fields detected:', numeric_fields)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback for demonstration

# Filter: keep rows with numeric_field > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with column '@id'={numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field (e.g., a categorical field)
categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
if categorical_fields:
    group_field_id = categorical_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped by {group_field_id} and calculated mean {numeric_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships using record set and field `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# If grouping possible, plot mean by group
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    grouped_df.sort_values(numeric_field_id, ascending=False).head(20).plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"mean({numeric_field_id})")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> protein mass spectrometry dataset via its Croissant schema, explored its record sets and fields using their `@id`, extracted and processed records as DataFrames, performed simple EDA including filtering, normalization, and grouping, and visualized field distributions—all powered by `mlcroissant`.

This notebook is a template—adapt as needed to your analytical use case by referencing record set and field `@id`s, and consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for advanced usage.
